# ICV P03: Results Checkpoint, complete registered evaluation

This notebook reuses four verified seed-43 checkpoints, trains the eight missing configuration/seed combinations (42 and 44), evaluates all 12 on validation, records an immutable protocol/checkpoint freeze, and then performs final clean test plus 27 fixed corruptions. The test split has 360 images. The final archive contains 336 unique checkpoint/condition evaluations and 120,960 paired prediction rows, not that many independent images.

**Before running:** enable **GPU T4** and **Internet**. Use **Add Input** to attach the **OUTPUT** of `datastr3am/icv-progress-check`, saved version **362621632**. Adding only the notebook code does not attach its weights. Alternatively create a private Kaggle input dataset from the supplied **icv_seed43_controls.zip** and attach it. Old `icv-second-approach` alone is insufficient: it lacks the camera-aware controls.

Use a fresh **Save Version > Save & Run All**. Default `RETRAIN_SEED43=False` trains only eight new runs. Before training, the four historical checkpoints and their environments are checked. Reviewed non-GPU packages are pinned; Kaggle CUDA packages are retained. A GPU/software compatibility mismatch stops instead of silently mixing incompatible training environments. `RETRAIN_SEED43=True` explicitly chooses twelve fresh runs and requires a new session; old results remain historical evidence.

**This is the final test stage.** `ACKNOWLEDGE_FINAL_TEST=True` is explicit in the first code cell. Save & Run All with this setting evaluates test only after the complete training/validation audits and recorded freeze. Do not tune training, labels, severities or seed selection after seeing test results. Set it to False before starting only if you intentionally want training/validation without test; then only the intermediate archive is produced.

Outputs: **icv_training_checkpoint.zip** is saved after freeze and before test. **icv_final_results.zip** is created only after all 12 x 28 evaluations, metric audits and comparisons complete. Send the final ZIP to the team. If evaluation fails, retain the intermediate ZIP and logs; do not repeat training just to repair evaluation. Existing evidence is never overwritten. CPU/local training is blocked.

The training protocol is unchanged: ImageNet-1K V1, full fine-tuning, batch 32, workers 2, AdamW lr/weight decay 1e-4, smoothing 0.05, warmup 3, cosine schedule, up to 50 epochs, patience 10, CUDA AMP. Earliest strict best clean-validation macro-F1 selects weights. Inference is FP32; latency uses batch 1, 10 warmups and 50 timed synchronized runs, excluding loading/preprocessing/host-to-device transfer.

The 27 fixed corruption conditions use `corruption_suite_v1` with the registered ranges; tensor hashes enforce identical evaluated views across models and training seeds. Mean +/- SD is across seeds; the registered 5,000-repeat paired image bootstrap is separate. No best seed or ensemble is selected. Failure candidates contain actual corrupted images and require human explanation. No training or results are published to GitHub by this notebook.


## 1. Source, environment and explicit final-test configuration


In [ ]:
import base64
import contextlib
import gc
import gzip
import hashlib
import importlib.metadata
import io
import json
import os
import shutil
import subprocess
import sys
import tarfile
import tomllib
from datetime import UTC, datetime
from pathlib import Path

from packaging.requirements import Requirement
from packaging.specifiers import SpecifierSet

SETUP_OK = CONTROLS_OK = DATA_OK = TESTS_OK = TRAINING_OK = VAL_OK = FROZEN_OK = TEST_OK = (
    SUMMARY_OK
) = False
# Default: reuse four checked seed-43 checkpoints, train eight missing runs.
RETRAIN_SEED43 = False
# Save & Run All with True authorizes final test inference after the recorded freeze.
# False runs only training/validation and produces icv_training_checkpoint.zip.
ACKNOWLEDGE_FINAL_TEST = True
checkpoints = {}
if not Path("/kaggle/working").is_dir():
    raise RuntimeError("Kaggle-only notebook. Local training is disabled.")

REPO_URL = "https://github.com/kujifined/Introduction-to-cv.git"
REPO_COMMIT = "760889adc87264122efe67d04e966dfeaaa319d8"
EXPECTED_MANIFEST_SHA256 = "df784b6590f90947db6fd40c8d2e9650ab168744a0bd400e13dc535927f8232d"
EXPECTED_ARCHIVE_SHA256 = "a85bcc6bfa1a40e76f28079e135694aaf5f6a4ee01203f7eaacff0ebafaa96f8"
PROJECT_ROOT = Path("/kaggle/temp/icv_final_repo")
ARTIFACT_ROOT = Path("/kaggle/working/icv_final_results")
TEMP_DATA = Path("/kaggle/temp/icv_final_data")
RESULTS_ZIP = Path("/kaggle/working/icv_final_results.zip")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["MPLCONFIGDIR"] = "/kaggle/temp/icv_final_mpl"


def run(command, *, cwd=PROJECT_ROOT):
    print("$", " ".join(str(item) for item in command), flush=True)
    subprocess.run([str(item) for item in command], cwd=cwd, check=True)


def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def save_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")


def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


if RESULTS_ZIP.exists() or (ARTIFACT_ROOT.exists() and any(ARTIFACT_ROOT.iterdir())):
    raise FileExistsError("Existing evidence is preserved. Start a fresh Save & Run All.")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", REPO_URL, PROJECT_ROOT], cwd=PROJECT_ROOT.parent)
    run(["git", "checkout", "--detach", REPO_COMMIT])
if not (PROJECT_ROOT / ".git").is_dir():
    raise RuntimeError("Project path is occupied by a non-checkout directory.")
revision = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if revision != REPO_COMMIT:
    raise RuntimeError("Unexpected source revision. Use a fresh Kaggle session.")
run(["git", "diff", "--exit-code", "HEAD"])
os.chdir(PROJECT_ROOT)
metadata = tomllib.loads((PROJECT_ROOT / "pyproject.toml").read_text())
python_version = ".".join(map(str, sys.version_info[:3]))
declared_range = metadata["project"]["requires-python"]
if python_version not in SpecifierSet(declared_range):
    raise RuntimeError(f"Python {python_version} is outside supported range {declared_range}.")

requirements = [Requirement(value) for value in metadata["project"]["dependencies"]]
requirements.extend([Requirement("gdown>=5.2,<6"), Requirement("pytest>=8.2,<10")])
# Match reviewed non-GPU packages when reusing historical controls.
if not RETRAIN_SEED43:
    reviewed = {"numpy": "2.1.3", "pillow": "12.3.0", "pandas": "2.3.3", "scikit-learn": "1.6.1"}
    requirements = [
        Requirement(f"{req.name}=={reviewed[req.name]}") if req.name in reviewed else req
        for req in requirements
    ]
gpu_names = {"torch", "torchvision"}
gpu_before = {name: installed_version(name) for name in gpu_names}
missing = []
for requirement in requirements:
    if requirement.marker is not None and not requirement.marker.evaluate():
        continue
    actual = installed_version(requirement.name)
    compatible = actual is not None and actual in requirement.specifier
    if requirement.name in gpu_names:
        if not compatible:
            raise RuntimeError(f"Incompatible Kaggle CUDA package: {requirement}; {actual}")
    elif not compatible:
        missing.append(str(requirement))
if missing:
    # Constraints also protect CUDA dependencies if pip needs a transitive package.
    constraint_file = Path("/kaggle/temp/icv_final_gpu_constraints.txt")
    constraint_file.write_text(
        "\n".join(f"{name}=={version}" for name, version in sorted(gpu_before.items())) + "\n"
    )
    run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "--only-binary=:all:",
            "--constraint",
            constraint_file,
            *missing,
        ]
    )
for requirement in requirements:
    if requirement.marker is not None and not requirement.marker.evaluate():
        continue
    actual = installed_version(requirement.name)
    if actual is None or actual not in requirement.specifier:
        raise RuntimeError(f"Dependency check failed: {requirement}; installed={actual}")
if gpu_before != {name: installed_version(name) for name in gpu_names}:
    raise RuntimeError("CUDA package versions unexpectedly changed.")


PAYLOAD_SHA256 = "bb4221cc4c2071c5c8bd644200a6c52b2d2a1569314d5b04bfe5b905c48f11eb"
PAYLOAD_BASE64 = (
    "..."
    "/wFd3R8ZAJABAA=="
)
EXPECTED_EXTENSION_FILES = {
    "tests/test_weekly_validation.py",
    "tests/test_results_checkpoint.py",
    "seed43_controls.json",
    "src/industrial_surface_defects/weekly_validation.py",
    "src/industrial_surface_defects/results_checkpoint.py",
}

embedded = base64.b64decode(PAYLOAD_BASE64, validate=True)
if hashlib.sha256(embedded).hexdigest() != PAYLOAD_SHA256:
    raise RuntimeError("Embedded code hash mismatch")
extension_hashes = {}
with tarfile.open(fileobj=io.BytesIO(gzip.decompress(embedded)), mode="r:") as archive:
    for member in archive.getmembers():
        if not member.isfile() or member.name not in EXPECTED_EXTENSION_FILES:
            raise RuntimeError("Unexpected embedded source entry")
        target = PROJECT_ROOT / member.name
        if target.exists():
            raise FileExistsError("Extension source already exists; start a fresh session")
        content = archive.extractfile(member).read()
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(content)
        extension_hashes[member.name] = hashlib.sha256(content).hexdigest()
if set(extension_hashes) != EXPECTED_EXTENSION_FILES:
    raise RuntimeError("Missing extension sources")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")

import torch  # noqa: E402 -- runtime dependencies are prepared first
from torchvision.models import EfficientNet_B0_Weights, ResNet18_Weights  # noqa: E402

import industrial_surface_defects  # noqa: E402
from industrial_surface_defects import results_checkpoint as final  # noqa: E402
from industrial_surface_defects.reproducibility import runtime_metadata  # noqa: E402

if not torch.cuda.is_available():
    raise RuntimeError("Enable Kaggle GPU T4; CPU/local training is disabled")
torch.cuda.set_device(0)
if (
    Path(industrial_surface_defects.__file__).resolve().parent
    != PROJECT_ROOT / "src/industrial_surface_defects"
):
    raise RuntimeError("Wrong cached project; restart the session")
if ResNet18_Weights.DEFAULT != ResNet18_Weights.IMAGENET1K_V1:
    raise RuntimeError("ResNet pretrained enum changed")
if EfficientNet_B0_Weights.DEFAULT != EfficientNet_B0_Weights.IMAGENET1K_V1:
    raise RuntimeError("EfficientNet pretrained enum changed")
ARTIFACT_ROOT.mkdir(parents=True)
current_runtime = runtime_metadata(torch.device("cuda"))
packages = {req.name: installed_version(req.name) for req in requirements}
environment = {
    "recorded_at_utc": datetime.now(UTC).isoformat(),
    "git_revision": revision,
    "runtime": current_runtime,
    "packages": packages,
    "declared_python_range": declared_range,
    "extension_payload_sha256": PAYLOAD_SHA256,
    "extension_files": extension_hashes,
    "retrain_seed43": RETRAIN_SEED43,
    "acknowledge_final_test": ACKNOWLEDGE_FINAL_TEST,
    "cuda_packages_retained": True,
}
save_json(ARTIFACT_ROOT / "cloud_environment.json", environment)
save_json(ARTIFACT_ROOT / "experiment_protocol.json", final.fixed_protocol())
for relative in EXPECTED_EXTENSION_FILES:
    target = ARTIFACT_ROOT / "reproducibility" / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(PROJECT_ROOT / relative, target)
SETUP_OK = True
print(json.dumps(environment, indent=2))

## 2. Import and verify the four historical checkpoints


In [ ]:
CONTROLS_OK = False
if not globals().get("SETUP_OK", False):
    raise RuntimeError("Setup must pass first")
configs = final.build_matrix_configs(PROJECT_ROOT, ARTIFACT_ROOT / "outputs")
expected_controls = json.loads((PROJECT_ROOT / "seed43_controls.json").read_text())
provenance = {}
if not RETRAIN_SEED43:
    provenance = final.import_seed43("/kaggle/input", ARTIFACT_ROOT, expected_controls)
    differences = final.check_control_environment(
        ARTIFACT_ROOT, current_runtime, packages, expected_controls
    )
    save_json(ARTIFACT_ROOT / "control_environment_check.json", {"differences": differences})
    if differences:
        print(json.dumps(differences, indent=2))
        raise RuntimeError(
            "Historical training environment differs. No training has started. "
            "To explicitly choose twelve fresh runs, set RETRAIN_SEED43=True and start "
            "a fresh Save & Run All. Do not silently mix incompatible training environments."
        )
    for name in final.CONFIGURATIONS:
        final.verify_training(
            ARTIFACT_ROOT / "outputs" / name / "seed_43/best.pt", configs[(name, 43)]
        )
CONTROLS_OK = True
print("CONTROLS READY. New trainings:", 12 if RETRAIN_SEED43 else 8)

## 3. Dataset integrity and all tests


In [ ]:
DATA_OK = TESTS_OK = False
if not all(globals().get(flag, False) for flag in ("SETUP_OK", "CONTROLS_OK")):
    raise RuntimeError("Setup failed; dataset preparation is blocked.")

import pandas as pd  # noqa: E402 -- dependency setup must finish first

from industrial_surface_defects.data import verify_manifest_files  # noqa: E402

manifest = PROJECT_ROOT / "data/manifests/neu_cls_v2.csv"
if sha256(manifest) != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError("Frozen manifest hash changed.")
run(["git", "diff", "--exit-code", "HEAD", "--", "data/manifests"])
TEMP_DATA.mkdir(parents=True, exist_ok=True)
archive = TEMP_DATA / "NEU-CLS.rar"
raw_root = TEMP_DATA / "raw"

decoder = shutil.which("unar")
if decoder is None:
    if shutil.which("apt-get") is None:
        raise RuntimeError("unar is required; no supported installer was found.")
    run(["apt-get", "-qq", "update"], cwd="/kaggle/working")
    run(
        ["apt-get", "-qq", "install", "-y", "--no-install-recommends", "unar"],
        cwd="/kaggle/working",
    )
    decoder = shutil.which("unar")
if decoder is None:
    raise RuntimeError("unar is unavailable after installation.")
if not archive.exists():
    run(
        [
            sys.executable,
            "scripts/download_neu_cls.py",
            "--acknowledge-source-terms",
            "--archive",
            archive,
        ]
    )
if sha256(archive) != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError("Official archive hash mismatch.")
if raw_root.exists() and (not raw_root.is_dir() or any(raw_root.iterdir())):
    raise FileExistsError("Extraction directory is occupied. Use a fresh run.")
raw_root.mkdir(parents=True, exist_ok=True)
extraction_command = [decoder, "-q", "-D", "-o", str(raw_root), str(archive)]
extraction = subprocess.run(
    extraction_command,
    cwd="/kaggle/working",
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    timeout=120,
    check=False,
)
(ARTIFACT_ROOT / "extraction.log").write_text(extraction.stdout, encoding="utf-8")
if extraction.returncode != 0 or not (raw_root / "NEU-CLS").is_dir():
    raise RuntimeError(f"unar extraction failed: {extraction.stdout[-1500:]}")
save_json(
    ARTIFACT_ROOT / "extraction_metadata.json",
    {
        "command": extraction_command,
        "archive_sha256": sha256(archive),
        "raw_data_in_results_zip": False,
    },
)
data_link = PROJECT_ROOT / "data/raw/NEU-CLS"
data_link.parent.mkdir(parents=True, exist_ok=True)
if not data_link.exists():
    data_link.symlink_to(raw_root, target_is_directory=True)
elif data_link.resolve() != raw_root.resolve():
    raise RuntimeError("Unexpected dataset directory; existing files will not be replaced.")
verify_manifest_files(manifest, data_link)
rows = pd.read_csv(manifest)
expected_counts = {"excluded": 1, "test": 360, "train": 1259, "val": 180}
if len(rows) != 1800 or rows.groupby("split").size().to_dict() != expected_counts:
    raise RuntimeError("Unexpected dataset or split counts.")
active = rows.loc[rows.status == "ok"]
if len(active) != 1799 or rows.path.duplicated().any():
    raise RuntimeError("Active dataset count or path uniqueness check failed.")
for key in ("sha256", "group_id"):
    if (active.groupby(key).split.nunique() > 1).any():
        raise RuntimeError(f"Known cross-split leakage: {key}")
if active.loc[active.split == "val"].groupby("class_name").size().tolist() != [30] * 6:
    raise RuntimeError("Unexpected validation class support.")
shutil.copyfile(manifest, ARTIFACT_ROOT / "neu_cls_v2.csv")
save_json(
    ARTIFACT_ROOT / "dataset_verification.json",
    {
        "manifest_sha256": sha256(manifest),
        "archive_sha256": sha256(archive),
        "original_files": 1800,
        "excluded_duplicates": 1,
        "active_images": 1799,
        "split_counts": expected_counts,
        "every_manifest_file_hash_verified": True,
        "active_paths_unique": True,
        "known_hash_and_group_splits_disjoint": True,
        "test_evaluation_performed": False,
    },
)
DATA_OK = True
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=PROJECT_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    check=False,
)
(ARTIFACT_ROOT / "pytest.log").write_text(test_result.stdout, encoding="utf-8")
print(test_result.stdout)
if test_result.returncode != 0:
    raise RuntimeError("Repository tests failed; training remains blocked.")
TESTS_OK = True
print("DATA AND TESTS OK")

## 4. Train the eight missing registered runs


In [ ]:
TRAINING_OK = VAL_OK = FROZEN_OK = TEST_OK = SUMMARY_OK = False
if not all(
    globals().get(flag, False) for flag in ("SETUP_OK", "CONTROLS_OK", "DATA_OK", "TESTS_OK")
):
    raise RuntimeError("Setup, controls, data integrity and all tests must pass")
if not Path("/kaggle/working").is_dir() or not torch.cuda.is_available():
    raise RuntimeError("Kaggle CUDA is required")
from industrial_surface_defects.train import train  # noqa: E402 -- guarded training


class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, value):
        for stream in self.streams:
            stream.write(value)
            stream.flush()
        return len(value)

    def flush(self):
        for stream in self.streams:
            stream.flush()


for (name, seed), config in configs.items():
    key = f"{name}/{seed}"
    if key in provenance:
        continue
    gc.collect()
    torch.cuda.empty_cache()
    print("TRAIN", name, "seed", seed, flush=True)
    with (
        (ARTIFACT_ROOT / f"{name}_seed{seed}_training.log").open("x") as log,
        contextlib.redirect_stdout(Tee(sys.stdout, log)),
        contextlib.redirect_stderr(Tee(sys.stderr, log)),
    ):
        checkpoint = train(config)
    final.verify_training(checkpoint, config)
    provenance[key] = {
        "origin": "fresh_this_kaggle_session",
        "trained_in_current_session": True,
        "original_training_runtime": current_runtime,
        "checkpoint_sha256": sha256(checkpoint),
    }
    save_json(ARTIFACT_ROOT / "checkpoint_provenance.json", provenance)
if len(provenance) != 12:
    raise RuntimeError("Incomplete registered training matrix")
TRAINING_OK = True
print("ALL 12 TRAINING RECORDS COMPLETE. No test inference has occurred.")

## 5. Evaluate and audit all twelve on validation


In [ ]:
VAL_OK = FROZEN_OK = TEST_OK = SUMMARY_OK = False
if not all(
    globals().get(flag, False)
    for flag in ("SETUP_OK", "CONTROLS_OK", "DATA_OK", "TESTS_OK", "TRAINING_OK")
):
    raise RuntimeError("All training records must pass before validation")
from industrial_surface_defects.evaluate import evaluate  # noqa: E402 -- validation only

for (name, seed), checkpoint in final.matrix_checkpoints(ARTIFACT_ROOT).items():
    gc.collect()
    torch.cuda.empty_cache()
    with (
        (ARTIFACT_ROOT / f"{name}_seed{seed}_validation.log").open("x") as log,
        contextlib.redirect_stdout(Tee(sys.stdout, log)),
        contextlib.redirect_stderr(Tee(sys.stderr, log)),
    ):
        evaluate(
            checkpoint,
            split="val",
            acknowledge_test_use=False,
            device_name="cuda",
            data_root=str(data_link),
            manifest_path=str(manifest),
            batch_size=32,
            num_workers=2,
            skip_latency=True,
        )
final.audit_validation(ARTIFACT_ROOT, configs, manifest)
VAL_OK = True
print("12 VALIDATION EVALUATIONS CHECKED. No checkpoint is selected using test results.")

## 6. Freeze protocol/checkpoints and save recovery archive


In [ ]:
FROZEN_OK = TEST_OK = SUMMARY_OK = False
if not all(
    globals().get(flag, False)
    for flag in ("SETUP_OK", "CONTROLS_OK", "DATA_OK", "TESTS_OK", "TRAINING_OK", "VAL_OK")
):
    raise RuntimeError("Full training and validation audits must pass before freeze")
final.write_freeze(ARTIFACT_ROOT, PROJECT_ROOT, configs, manifest)
checkpoint_zip = Path("/kaggle/working/icv_training_checkpoint.zip")
checkpoint_digest = final.export_bundle(
    ARTIFACT_ROOT, checkpoint_zip, status="training_validation_frozen_test_not_started"
)
FROZEN_OK = True
print("FROZEN BEFORE TEST:", sha256(ARTIFACT_ROOT / "protocol_freeze.json"))
print("Recovery archive:", checkpoint_zip, "SHA256:", checkpoint_digest)

## 7. Final test and full fixed corruption suite


In [ ]:
TEST_OK = SUMMARY_OK = False
if not all(
    globals().get(flag, False)
    for flag in (
        "SETUP_OK",
        "CONTROLS_OK",
        "DATA_OK",
        "TESTS_OK",
        "TRAINING_OK",
        "VAL_OK",
        "FROZEN_OK",
    )
):
    raise RuntimeError("Every pre-test stage must pass")
if not ACKNOWLEDGE_FINAL_TEST:
    print("TEST STAYS LOCKED. Download icv_training_checkpoint.zip; no final results claimed.")
else:
    with (
        (ARTIFACT_ROOT / "final_test_evaluation.log").open("x") as log,
        contextlib.redirect_stdout(Tee(sys.stdout, log)),
        contextlib.redirect_stderr(Tee(sys.stderr, log)),
    ):
        final.evaluate_test_matrix(
            ARTIFACT_ROOT,
            PROJECT_ROOT,
            manifest,
            data_link,
            acknowledge_test_use=True,
        )
    TEST_OK = True
    print("12 CHECKPOINTS x 28 TEST CONDITIONS COMPLETE AND INDEPENDENTLY AUDITED")

## 8. Comparisons, runtime, failure candidates and final archive


In [ ]:
SUMMARY_OK = False
if not ACKNOWLEDGE_FINAL_TEST:
    if not globals().get("FROZEN_OK", False):
        raise RuntimeError("No completed training bundle")
    print("Training/validation only. Final test, comparison and report reviews are pending.")
else:
    if not globals().get("TEST_OK", False):
        raise RuntimeError("Final test incomplete; no final-results archive will be created")
    final.assert_frozen(ARTIFACT_ROOT, PROJECT_ROOT, manifest, acknowledge_test_use=True)
    with (
        (ARTIFACT_ROOT / "aggregation.log").open("x") as log,
        contextlib.redirect_stdout(Tee(sys.stdout, log)),
        contextlib.redirect_stderr(Tee(sys.stderr, log)),
    ):
        final.aggregate_results(ARTIFACT_ROOT)
        final.failure_materials(ARTIFACT_ROOT, data_link)
    (ARTIFACT_ROOT / "REVIEW_README.md").write_text(
        "# Results Checkpoint: recorded final evaluation\n\n"
        "All 12 registered checkpoints and 28 fixed test conditions were evaluated. "
        "Use clean_scores_by_configuration.csv, comparison_robustness/, runtime_summary.csv "
        "and test_severity_curves.png. Full rows are in test_scores_all_conditions.csv.\n\n"
        "Individual predictions, metrics, independent audits, "
        "per-class tables and confusion matrices "
        "are under outputs/<configuration>/seed_<seed>/evaluation/. Clean is evaluation/test; "
        "corrupted inputs are evaluation/corruptions/<condition>. "
        "The combined per-run suite is evaluation/robustness/.\n\n"
        "protocol_freeze.json and its SHA record the pre-test choices and checkpoint identities. "
        "checkpoint_provenance.json distinguishes historical and new training. "
        "Training-stage times span sessions; latency is FP32 batch-one "
        "in this evaluation session.\n\n"
        "observed_test_errors.csv retains every model/seed/condition error. "
        "failure_candidates.csv/png shortlist distinct source images, showing original and actual "
        "corrupted images. These candidates need human explanations; no mechanisms are asserted. "
        "If fewer than three distinct errors exist, report that honestly.\n\n"
        "Human post-run reviews remain pending. Repeated seeds and transformed views are paired "
        "on 360 source crops, not independent extra samples. No GitHub publication is performed.\n"
    )
    final.assert_frozen(ARTIFACT_ROOT, PROJECT_ROOT, manifest, acknowledge_test_use=True)
    digest = final.export_bundle(
        ARTIFACT_ROOT, RESULTS_ZIP, status="registered_final_test_evaluation_complete"
    )
    SUMMARY_OK = True
    print("Finished. Download", RESULTS_ZIP)
    print("Archive SHA256:", digest)
    print("Send the same final archive to all three reviewers. Human reviews remain pending.")